# 02 — Body-weight EDA
Descriptive review of measurement levels, changes, frequency, causal rolling trend, and potential outliers. No causal interpretation is made.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from metaform_ds.features import add_body_weight_features
from metaform_ds.quality import build_quality_report
from metaform_ds.utils.notebooks import load_notebook_data
data = load_notebook_data(); weight = add_body_weight_features(data.body_weight)
print(f'DATA MODE: {data.mode}; rows: {len(weight)}')
if data.mode == 'SYNTHETIC': print('SYNTHETIC demonstration only.')

In [ ]:
if weight.empty:
    print('Insufficient data: no body measurements are available.')
else:
    print(weight['weight_kg'].describe().to_string())
    print(build_quality_report(weight, dataset_name='body_weight', timestamp_column='measured_at', duplicate_timestamp_keys=['anonymous_user_id', 'measured_at']).issues.to_string(index=False))

In [ ]:
if not weight.empty:
    first_user = weight['anonymous_user_id'].iloc[0]; view = weight[weight['anonymous_user_id'].eq(first_user)]
    fig, ax = plt.subplots(); ax.plot(view['measured_at'], view['weight_kg'], marker='o'); ax.set(title='Body weight over time', xlabel='Measurement date (UTC)', ylabel='Weight (kg)'); fig.autofmt_xdate(); plt.show()

In [ ]:
changes = weight.groupby('anonymous_user_id')['weight_kg'].diff().dropna()
if changes.empty: print('Insufficient data for a change distribution.')
else:
    fig, ax = plt.subplots(); ax.hist(changes, bins=min(12, max(3, len(changes)))); ax.set(title='Between-measurement weight changes', xlabel='Change (kg)', ylabel='Measurements'); plt.show()

In [ ]:
intervals = weight.groupby('anonymous_user_id')['measured_at'].diff().dt.total_seconds().div(86400).dropna()
if intervals.empty: print('Insufficient data for measurement-frequency analysis.')
else:
    fig, ax = plt.subplots(); ax.hist(intervals, bins=min(12, max(3, len(intervals)))); ax.set(title='Measurement intervals', xlabel='Days between measurements', ylabel='Intervals'); plt.show()

In [ ]:
if not weight.empty:
    view = weight[weight['anonymous_user_id'].eq(weight['anonymous_user_id'].iloc[0])]
    fig, ax = plt.subplots(); ax.plot(view['measured_at'], view['rolling_mean_7']); ax.set(title='Past-only 7-observation rolling weight', xlabel='Measurement date (UTC)', ylabel='Prior rolling mean (kg)'); fig.autofmt_xdate(); plt.show()

In [ ]:
if len(weight) < 4: print('Insufficient data for robust outlier review.')
else:
    q1, q3 = weight['weight_kg'].quantile([0.25, 0.75]); iqr = q3 - q1; outliers = weight[(weight['weight_kg'] < q1 - 1.5*iqr) | (weight['weight_kg'] > q3 + 1.5*iqr)]
    print(f'IQR-flagged observations for review: {len(outliers)}. Flags are not automatic deletions.')
print('Sufficiency depends on history length, cadence, missingness, and the future prediction horizon; this notebook does not declare the data model-ready.')